# Exploratory Data Analysis
This notebook mirrors the outputs of `src/eda.py`, demonstrating the insights gained from the MTPL2 dataset.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Apply styles
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("paper", font_scale=1.2)
sns.set_palette("muted")

# Load data
df = pd.read_csv('../data/processed/features.csv')
df.head()

## 1. Claim Frequency by Region
Regions R82 and R24 show the highest claim frequencies, suggesting potential urban concentration.

In [ ]:
plt.figure(figsize=(10, 6))
grouped = df.groupby("Region").agg(claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
grouped["freq"] = grouped["claims"] / grouped["exposure"]
grouped = grouped.sort_values("freq", ascending=False)
sns.barplot(x=grouped.index, y=grouped["freq"], color="steelblue")
plt.title("Claim Frequency by Region")
plt.ylabel("Claims per Year of Exposure")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Claim Frequency by Driver Age Band
The 'young driver' risk premium is strongly justified, as the 18-25 age band exhibits the highest frequency.

In [ ]:
plt.figure(figsize=(8, 5))
grouped = df.groupby("DrivAgeBand", observed=True).agg(claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
grouped["freq"] = grouped["claims"] / grouped["exposure"]
sns.barplot(x=grouped.index, y=grouped["freq"], color="mediumseagreen")
plt.title("Claim Frequency by Driver Age Band")
plt.ylabel("Claims per Year")
plt.tight_layout()
plt.show()

## 3. Claim Frequency by BonusMalus Decile
BonusMalus is highly predictive of future claim frequency, with a steep, monotonic increase across deciles.

In [ ]:
plt.figure(figsize=(8, 5))
df_bm = df.copy()
df_bm["BM_Decile"] = pd.qcut(df_bm["BonusMalus"], q=10, duplicates="drop")
grouped = df_bm.groupby("BM_Decile", observed=True).agg(claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
grouped["freq"] = grouped["claims"] / grouped["exposure"]
x_labels = [str(i) for i in grouped.index]
plt.plot(x_labels, grouped["freq"], marker="o", linestyle="-", color="purple")
plt.title("Claim Frequency by BonusMalus Decile")
plt.ylabel("Claims per Year")
plt.xlabel("BonusMalus Deciles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Severity Distribution
Claim amounts follow a heavy-tailed distribution, heavily skewed to the right.

In [ ]:
plt.figure(figsize=(8, 5))
claims = df[df["ClaimAmount_total"] > 0]["ClaimAmount_total"]
sns.histplot(np.log10(claims), bins=50, color="indianred")
plt.title("Distribution of Claim Amounts (Log10 Scale)")
plt.xlabel("Log10(Claim Amount in Euros)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()